In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

from xgboost import XGBClassifier

from transformers import DistilBertTokenizer, DistilBertModel
import torch

In [2]:
from datasets import load_dataset

dataset = load_dataset("vargr/main_instagram")

df = dataset['train'].to_pandas()
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-b6868cb1ddbd5c(…):   0%|          | 0.00/159M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/605868 [00:00<?, ? examples/s]

,sid,sid_profile,shortcode,profile_id,date,post_type,description,likes,comments,username,bio,following,followers,num_posts,is_business_account,lang,description_category,description_grade,image_grade,path
0,8304346,3496776,Bt5LFpZlm3z,2237947779,2019-02-15 08:02:35,1,Laps in spring like conditions. Getting these ...,150,3,andylund_,"Professional Bicycle technician, Intense Racin...",520,1204,494,False,en,sports,0.309194,2.692171,z/3/Bt5LFpZlm3z.jpg
1,13034348,4376375,BvNGiDVg_6P,6517756,2019-03-19 22:18:58,1,It’s pink drink time!! 😋 @starbucks,15,0,eyespy0099,South Florida. Check out my YouTube channel fo...,1996,1263,4647,False,en,food_&_dining,0.497623,2.289400,p/6/BvNGiDVg_6P.jpg
2,17030765,3585466,BxaRBOGn9g-,13110141824,2019-05-13 19:03:32,1,C•A•R•B•O•N•A•R•A🍝🍝🍝 #pasta #restaurant #itali...,41,3,mimafoodiebcn,"Hi! 👋🏼 I’m Mima, Currently eating in Barcelona...",212,139,9,True,en,food_&_dining,0.340083,3.475924,-/g/BxaRBOGn9g-.jpg
3,8691830,-1,BFHVYmNv1pR,205039053,2016-05-07 19:34:45,1,Suited up for the Leeds Beckett Sports Awards ...,292,1,yonakw,🇯🇲 Diver\nSupported by @levirootsmusic\nSmuggl...,445,28163,1146,True,en,sports,0.067614,3.365403,r/p/BFHVYmNv1pR.jpg
4,13009836,4368355,Bv10nn0FgfA,9713615,2019-04-04 18:52:40,2,In the middle of it at Dundee! ❤️ What great v...,454,11,darude,🇫🇮 Proud Finn\n🎧 Producer/DJ\n⚡️My new single ...,459,27753,1621,True,en,diaries_&_daily_life,0.233947,2.447611,a/f/Bv10nn0FgfA.jpg


In [3]:
df = df[df['followers'] > 0]
df = df.dropna(subset=['description'])

In [4]:
df['engagement_rate'] = (df['likes'] + df['comments']) / df['followers']

In [5]:
def classify_engagement(rate):

    if rate < 0.05:
        return 0

    elif rate < 0.15:
        return 1

    else:
        return 2


df['engagement_class'] = df['engagement_rate'].apply(classify_engagement)

In [6]:
df['date'] = pd.to_datetime(df['date'])

df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek

In [7]:
df['followers'] = np.log1p(df['followers'])
df['following'] = np.log1p(df['following'])
df['num_posts'] = np.log1p(df['num_posts'])

In [11]:
metadata_columns = [
    'followers',
    'following',
    'num_posts',
    'hour',
    'day_of_week',
    'is_business_account',
    'description_grade',
    'image_grade',
    'post_type'
]

meta_features = df[metadata_columns].values

In [12]:
scaler = StandardScaler()

meta_features = scaler.fit_transform(meta_features)

In [13]:
y = df['engagement_class']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    meta_features,
    y,
    test_size=0.2,
    random_state=42
)

In [15]:
model = XGBClassifier()

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [16]:
preds = model.predict(X_test)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.70      0.59      0.64     38227
           1       0.54      0.64      0.58     49737
           2       0.62      0.56      0.59     33164

    accuracy                           0.60    121128
   macro avg       0.62      0.60      0.60    121128
weighted avg       0.61      0.60      0.60    121128



In [17]:
def engagement_range(class_label, followers):

    if class_label == 0:
        return followers * 0.01, followers * 0.05

    elif class_label == 1:
        return followers * 0.05, followers * 0.15

    else:
        return followers * 0.15, followers * 0.30

In [18]:
def reach_estimator(class_label):

    if class_label == 0:
        return "Limited reach (mostly followers)"

    elif class_label == 1:
        return "Moderate reach"

    else:
        return "High reach (likely shown to non-followers)"

In [19]:
best_times = df.groupby('hour')['engagement_rate'].mean().sort_values(ascending=False)

best_times.head(5)

,engagement_rate
hour,
4,0.135646
3,0.133309
6,0.132618
5,0.132204
2,0.131121


In [20]:
import re

def extract_hashtags(text):

    return re.findall(r"#(\w+)", text)


df['hashtags'] = df['description'].apply(extract_hashtags)

In [21]:
from collections import Counter

all_tags = Counter([tag for tags in df['hashtags'] for tag in tags])

all_tags.most_common(10)

[('love', 11354),
 ('london', 11191),
 ('nyc', 10607),
 ('photography', 9600),
 ('travel', 8558),
 ('instagood', 7417),
 ('photooftheday', 7099),
 ('nature', 6727),
 ('food', 6559),
 ('foodie', 5820)]